<img src="./nttdata_logo" width="200">


# Delta Table

Delta Tableの基本的な概念と、作成方法を説明する。

##### 目標
1. SQLクエリを実行する
1. スキーマを作成する
1. 管理テーブル・外部テーブルを理解する
1. テーブルをCTAS、CSVから作成する
1. テーブルに制約をつける
1. テーブルに行を追加・削除する
1. テーブルのバージョンを管理する




## 実行方法
Spark SQLは、複数のインターフェースを持つ構造化データ処理のモジュールです。

Spark SQLとは2つの方法でやり取りすることができます：
1. SQLクエリを実行する
1. DataFrame APIを利用する

In [0]:
%sql
SELECT customerID, first_name, last_name
FROM samples.bakehouse.sales_customers

In [0]:
display(spark
        .table("samples.bakehouse.sales_customers")
        .select("customerID", "first_name", "last_name")
       )

## Databricks上でのテーブルの扱い

Apache Sparkをはじめとするデータレイク系のフレームワークでは、ストレージ上にCSV, JSON, Parquet, DeltaLake, Avroなどのファイルとして保存しておいて、それらをプログラム上でテーブル形式にして扱うことが一般的です。例えば、SparkのPython APIであるPySparkだと、df = spark.read.csv('/path/to/file.csv')のようにデータフレームとして読み込ませることができ、CSVファイルをそのままテーブル形式として扱うことができます。つまり、データを「ファイル」として扱うことが基本的になります。

一方で、テーブルを扱うなら、特に検索や集計処理をする場合であれば、SQLを使いたくなるのは自然な流れだと思います。ただし、SQLは「ファイル」ではなく「テーブル名」としてデータ参照を行う必要があります。

このギャップを埋めるために、Apache SparkではMetastoreと呼ばれるサービス上で「テーブル名」と「ファイル」の紐付けができる様になっています。このMetastoreサービスのオプションとしては、ドライバノードローカル(Derby), Hive Metastore, Unity Catalog(Databricksプロダクト)などがあります(Sparkのシステムの環境設定で定義されています)。もちろん、SQLのテーブルとして使用するため、テーブル名に対応するファイル保存先パスの他にも、カラム名などのスキーマ情報や、テーブル設定(テーブルプロパティ)なども同時に記録管理されます。

以上、まとめると、Apache Sparkではデータ保存(永続化)とその参照の枠組みとして、データの実態はファイルとしてストレージ上に保存し、そのデータに紐づくテーブル名やスキーマ情報などはMetastoreに保存されます。つまり、ユーザーは、データの実体とテーブル・メタデータの2つを意識する必要があります。

![image_1772953082795.png](./image_1772953082795.png "image_1772953082795.png")


 
## スキーマ(データベース)
まず、スキーマ（データベース）を作成しましょう。カタログを別タブで開きながら確認してください。


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.test;

In [0]:
%sql
DESCRIBE SCHEMA EXTENDED workspace.test;

## テーブルの種類

### 管理テーブル
マネージドテーブル は、ユーザーがテーブル名だけを使ってデータ(Sparkデータフレーム)を永続化する方法で、データの保存先(ストレージとパス)はSparkシステム側で自動的にアサインされます。つまり、ユーザーは保存先などストレージ側を意識する必要がありません。データを削除する場合も、DROP TABLE文を実行するだけで済み、テーブルのメタデータの削除と、データ実体の削除が同時に実行されます。データの保存先が自動管理されるので 「マネージド」テーブル と呼びます。

### 外部テーブル
一方、アンマネージドテーブルは、ユーザーがデータの保存先パスを指定して永続化し、後からテーブル名とデータ保存先をリンクします。この場合、データの削除には、DROP TABLE文でのテーブル(のメタデータ)削除に加えて、データ保存先のデータ実体を削除する必要があります。つまり、ユーザーはテーブル(メタデータ)とデータ実体の保存先の両方を意識する必要があり、よって、「アンマネージド」テーブルと呼ばれています。この方法だと、Sparkシステム管理のストレージ外上に置かれているデータに対してもテーブル名とリンクができるため、この意味で、アンマネージドテーブルは「外部テーブル(External Table)」 とも呼ばれます。

## 管理テーブル
場所のパスを指定しないことで、 **管理** テーブルを作成します。

テーブルは、前述のスキーマ（データベース）内に作成します。

データのカラムとデータ型を推測するデータがないため、テーブルのスキーマを定義する必要があります。

In [0]:
%sql
USE  workspace.test;

CREATE OR REPLACE TABLE managed_table (width INT, length INT, height INT);
INSERT INTO managed_table VALUES (3, 2, 1);
SELECT * FROM managed_table;

In [0]:
%sql
DROP TABLE managed_table;


 
テーブルのディレクトリとそのログファイルおよびデータファイルは削除されます。スキーマ（データベース）ディレクトリだけが残ります。

ここでは外部テーブルは作成しませんが、管理テーブルと外部テーブルがあることは、頭の中に入れておいてください。



## SELECTを使用したテーブルの作成（CTAS）

**`CREATE TABLE AS SELECT`** ステートメントは、入力クエリから取得したデータを使用してDeltaテーブルを作成し、データを追加します。


In [0]:
%sql
CREATE OR REPLACE TABLE customers_summary AS
SELECT customerID, first_name, last_name
FROM samples.bakehouse.sales_customers
LIMIT 10;

SELECT * FROM customers_summary;


## CSVからのテーブルの作成

CSVを一時ビューにし、その一時ビューからCTASでDeltaテーブルを作成。



In [0]:
%sql
CREATE OR REPLACE TEMP VIEW tmp_vw
  (id LONG, name STRING, age INTEGER)
USING CSV
OPTIONS (
  path = "/Workspace/Users/kenta.owada@jp.nttdata.com/databricks-learn/csv/sample.csv",
  header = "true",
  delimiter = ","
);

CREATE TABLE delta_table AS
  SELECT * FROM tmp_vw;
  
SELECT * FROM delta_table;


## テーブル制約の追加

上記のエラーメッセージは、 **`CHECK制約`** を指しています。生成された列は、チェック制約の特別な実装です。

Delta Lakeはスキーマの書き込み時に強制するため、Databricksはテーブルに追加されるデータの品質と整合性を確保するための標準SQL制約管理節をサポートできます。

Databricksは現在、2つのタイプの制約をサポートしています:
* <a href="https://docs.databricks.com/delta/delta-constraints.html#not-null-constraint" target="_blank">**`NOT NULL`** 制約</a>
* <a href="https://docs.databricks.com/delta/delta-constraints.html#check-constraint" target="_blank">**`CHECK`** 制約</a>

どちらの場合も、制約を定義する前に、制約に違反するデータが既にテーブルに存在しないことを確認する必要があります。一度制約がテーブルに追加されると、制約に違反するデータは書き込みエラーを引き起こします。


In [0]:
%sql
ALTER TABLE delta_table ADD CONSTRAINT valid_date CHECK (age > '0');


## 行の追加

**`INSERT INTO`** を使用して、既存のDeltaテーブルに新しい行を原子的に追加できます。これにより、毎回上書きするよりも効率的な既存のテーブルへの増分更新が可能になります。

**`INSERT INTO`** を使用して、**`sales`** テーブルに新しい販売レコードを追加します。


In [0]:
%sql
INSERT INTO delta_table
SELECT  customerID, first_name, "20" FROM customers_summary;

SELECT * FROM delta_table;

In [0]:
%sql
SELECT * FROM delta_table;


## マージ更新

**`MERGE`** SQL操作を使用して、ソーステーブル、ビュー、またはDataFrameからターゲットDeltaテーブルにデータをアップサートできます。 Delta Lakeは **`MERGE`** での挿入、更新、削除をサポートし、高度なユースケースを簡素化するためのSQL標準を超えた拡張構文をサポートしています。

<strong><code>
MERGE INTO target a<br/>
USING source b<br/>
ON {merge_condition}<br/>
WHEN MATCHED THEN {matched_action}<br/>
WHEN NOT MATCHED THEN {not_matched_action}<br/>
</code></strong>

一般的なETLユースケースは、ログや他の絶えず増加するデータセットを、一連の追加操作を通じてDeltaテーブルに収集することです。

多くのソースシステムは重複レコードを生成することがあります。マージを使用すると、デュプリケートレコードを挿入せずに、インサート専用のマージを実行して回避できます。



 
**`MERGE`** の主な利点：
* 更新、挿入、削除が1つのトランザクションとして完了します
* マッチングフィールドに加えて複数の条件を追加できます
* カスタムロジックを実装するための幅広いオプションを提供します


In [0]:
%sql
CREATE OR REPLACE TABLE merge_table (id LONG,name STRING,age INTEGER);
INSERT INTO merge_table VALUES (2, "yuto", 40);
INSERT INTO merge_table VALUES (3, "ken", 50);


In [0]:
%sql
MERGE INTO delta_table a
USING merge_table b
ON a.id = b.id
WHEN MATCHED AND b.id IS NOT NULL THEN
  UPDATE SET name = b.name, age = b.age
WHEN NOT MATCHED THEN INSERT *;

SELECT * FROM delta_table;



## ヒストリーを持つDeltaテーブルの作成

このクエリの実行を待っている間、実行されているトランザクションの合計数を特定できるか試してみてください。


In [0]:
%sql
CREATE TABLE students
  (id INT, name STRING, value DOUBLE);
  
INSERT INTO students VALUES (1, "Yve", 1.0);
INSERT INTO students VALUES (2, "Omar", 2.5);
INSERT INTO students VALUES (3, "Elia", 3.3);

INSERT INTO students
VALUES 
  (4, "Ted", 4.7),
  (5, "Tiffany", 5.5),
  (6, "Vini", 6.3);
  
UPDATE students 
SET value = value + 1
WHERE name LIKE "T%";

DELETE FROM students 
WHERE value > 6;

CREATE OR REPLACE TEMP VIEW updates(id, name, value, type) AS VALUES
  (2, "Omar", 15.2, "update"),
  (3, "", null, "delete"),
  (7, "Blue", 7.7, "insert"),
  (11, "Diya", 8.8, "update");
  
MERGE INTO students b
USING updates u
ON b.id=u.id
WHEN MATCHED AND u.type = "update"
  THEN UPDATE SET *
WHEN MATCHED AND u.type = "delete"
  THEN DELETE
WHEN NOT MATCHED AND u.type = "insert"
  THEN INSERT *;


## Delta Lakeトランザクションの確認

Delta Lakeテーブルへのすべての変更はトランザクションログに保存されているため、簡単に<a href="https://docs.databricks.com/spark/2.x/spark-sql/language-manual/describe-history.html" target="_blank">テーブルの履歴</a>を確認できます。


In [0]:
%sql
DESCRIBE HISTORY students

In [0]:
%sql
SELECT * 
FROM students VERSION AS OF 3


## バージョンをロールバックする

テーブルから一部のレコードを手動で削除するクエリを入力しようとして、次の状態でこのクエリを誤って実行したと仮定しましょう。

**`RESTORE`** <a href="https://docs.databricks.com/spark/latest/spark-sql/language-manual/delta-restore.html" target="_blank">コマンド</a> はトランザクションとして記録されます。テーブルのすべてのレコードを誤って削除した事実を完全に隠すことはできませんが、操作を元に戻し、テーブルを望ましい状態に戻すことはできます。



In [0]:
%sql
DELETE FROM students;
SELECT * FROM students;

In [0]:
%sql
RESTORE TABLE students TO VERSION AS OF 8;
SELECT * FROM students;


## 古いファイルをクリーンアップする

DatabricksはDelta Lakeテーブル内の古いログファイル（デフォルトでは30日以上）を自動的にクリーンアップします。
チェックポイントが書き込まれるたびに、Databricksはこの保持期間よりも古いログエントリを自動的にクリーンアップします。

Delta Lakeのバージョニングとタイムトラベルは、最新バージョンのクエリとクエリのロールバックには適していますが、大規模な本番テーブルのすべてのバージョンのデータファイルを無期限に保持することは非常に高価です（また、PIIが存在する場合、コンプライアンスの問題につながる可能性があります）。

古いデータファイルを手動で削除する場合、これは **`VACUUM`** 操作で実行できます。

次のセルのコメントを解除し、 **`0 HOURS`** の保持期間で実行して、現在のバージョンのみを保持できます：


In [0]:
%sql
VACUUM students RETAIN 3 HOURS